# 02 — API Ingestion → PostgreSQL

Pipeline kéo dữ liệu từ 4 nguồn API và đẩy vào PostgreSQL.

## Luồng hoạt động
```
FreeGoldAPI ─┐
yfinance ────┼──► raw.gold_prices / raw.yfinance_daily
FRED API ────┼──► raw.fred_daily / raw.fred_monthly
EIA API ─────┘──► raw.eia_oil
                        │
                        ▼
              staging.daily_master (SQL populate)
                        │
                        ▼
              features.* (SQL feature engineering)
```


In [1]:
# ── 0. Setup ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Thêm project root vào sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from src.utils.logging_config import get_logger
logger = get_logger('notebook.02_ingestion')
logger.info('=== Notebook 02: API Ingestion bắt đầu ===')

2026-05-05 00:28:30 | INFO     | notebook.02_ingestion | === Notebook 02: API Ingestion bắt đầu ===


In [2]:
# ── 1. Tạo Schema và Tables (chạy 1 lần) ─────────────────────────────────
from src.data.storage.postgres_client import run_schema_pipeline, get_row_count

print('[1/5] Tạo database schema...')
run_schema_pipeline()
print('✓ Schema tạo xong (raw, staging, features)')

[1/5] Tạo database schema...
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | Schema đảm bảo tồn tại: raw
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | Schema đảm bảo tồn tại: staging
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | Schema đảm bảo tồn tại: features
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | Bắt đầu pipeline [schema]
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | [schema] [1/3] Chạy: 01_raw_tables.sql
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | [schema] [2/3] Chạy: 02_staging_tables.sql
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | [schema] [3/3] Chạy: 03_feature_tables.sql
2026-05-05 00:28:34 | INFO     | src.data.storage.postgres_client | SQL file ex

In [3]:
# ── 2. Ingest Gold Prices ─────────────────────────────────────────────────
# FreeGoldAPI (lịch sử) + yfinance GC=F (cập nhật đến hôm nay)
from src.data.ingestion.freegold_ingestion import ingest_gold_prices

print('[2/5] Ingesting gold prices...')
n = ingest_gold_prices(start='2000-01-01')
count = get_row_count('raw', 'gold_prices')
print(f'✓ Gold prices: {n} upserted | raw.gold_prices total: {count:,} rows')

[2/5] Ingesting gold prices...
2026-05-05 00:28:36 | INFO     | src.data.ingestion.freegold_ingestion | Fetching FreeGoldAPI
2026-05-05 00:28:37 | INFO     | src.data.ingestion.freegold_ingestion | FreeGoldAPI fetched
2026-05-05 00:28:37 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-05 00:28:37 | ERROR    | src.data.ingestion.freegold_ingestion | Lỗi ingest FreeGoldAPI
Traceback (most recent call last):
  File "e:\study_again\gold_time_prediction\src\data\ingestion\freegold_ingestion.py", line 165, in ingest_gold_prices
    n = upsert_dataframe(
        ^^^^^^^^^^^^^^^^^
  File "e:\study_again\gold_time_prediction\src\data\storage\postgres_client.py", line 144, in upsert_dataframe
    execute_values(cur, sql, rows, template=None, page_size=1000)
  File "c:\Users\Tuan\anaconda3\envs\std_again\Lib\site-packages\psycopg2\extras.py", line 1299, in execute_values
    cur.execute(b''.join(parts))
psycopg2.errors.UndefinedColumn: column "index" of relation "gold_pr

In [4]:
# ── 3. Ingest yfinance OHLCV ──────────────────────────────────────────────
# DXY, Silver, S&P500, VIX, Oil, Treasury ETF...
from src.data.ingestion.yfinance_ingestion import ingest_yfinance_all

print('[3/5] Ingesting yfinance OHLCV...')
yf_results = ingest_yfinance_all(start='2000-01-01')

# Hiển thị kết quả
import pandas as pd
df_yf_summary = pd.DataFrame([
    {'ticker': k, 'rows_upserted': v, 'status': 'OK' if v > 0 else ('EMPTY' if v == 0 else 'ERROR')}
    for k, v in yf_results.items()
])
print(df_yf_summary.to_string(index=False))

count = get_row_count('raw', 'yfinance_daily')
print(f'\n✓ raw.yfinance_daily total: {count:,} rows')

[3/5] Ingesting yfinance OHLCV...
2026-05-05 00:28:42 | INFO     | src.data.ingestion.yfinance_ingestion | Fetching yfinance OHLCV
2026-05-05 00:28:44 | INFO     | src.data.ingestion.yfinance_ingestion | yfinance OHLCV fetched
2026-05-05 00:28:44 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-05 00:28:44 | ERROR    | src.data.ingestion.yfinance_ingestion | Lỗi ingest yfinance ticker
Traceback (most recent call last):
  File "e:\study_again\gold_time_prediction\src\data\ingestion\yfinance_ingestion.py", line 129, in ingest_yfinance_all
    n = upsert_dataframe(
        ^^^^^^^^^^^^^^^^^
  File "e:\study_again\gold_time_prediction\src\data\storage\postgres_client.py", line 144, in upsert_dataframe
    execute_values(cur, sql, rows, template=None, page_size=1000)
  File "c:\Users\Tuan\anaconda3\envs\std_again\Lib\site-packages\psycopg2\extras.py", line 1299, in execute_values
    cur.execute(b''.join(parts))
psycopg2.errors.UndefinedColumn: column "index" of rel

In [5]:
# ── 4. Ingest FRED ────────────────────────────────────────────────────────
from src.data.ingestion.fred_ingestion import ingest_fred_daily, ingest_fred_monthly

print('[4a/5] Ingesting FRED daily series...')
fred_d = ingest_fred_daily(start='2000-01-01')
count_d = get_row_count('raw', 'fred_daily')
print(f'✓ FRED daily: raw.fred_daily total {count_d:,} rows')

print('[4b/5] Ingesting FRED monthly series...')
fred_m = ingest_fred_monthly(start='2000-01-01')
count_m = get_row_count('raw', 'fred_monthly')
print(f'✓ FRED monthly: raw.fred_monthly total {count_m:,} rows')

[4a/5] Ingesting FRED daily series...
2026-05-05 00:29:02 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-05 00:29:02 | ERROR    | src.data.ingestion.fred_ingestion | Lỗi FRED daily
Traceback (most recent call last):
  File "e:\study_again\gold_time_prediction\src\data\ingestion\fred_ingestion.py", line 63, in ingest_fred_daily
    n = upsert_dataframe(df, table="fred_daily", schema=PG_SCHEMA_RAW, conflict_cols=["date", "series_id"])
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\study_again\gold_time_prediction\src\data\storage\postgres_client.py", line 144, in upsert_dataframe
    execute_values(cur, sql, rows, template=None, page_size=1000)
  File "c:\Users\Tuan\anaconda3\envs\std_again\Lib\site-packages\psycopg2\extras.py", line 1299, in execute_values
    cur.execute(b''.join(parts))
psycopg2.errors.UndefinedColumn: column "index" of relation "fred_daily" does not exist
LINE 1: INSERT

In [6]:
# ── 5. Ingest EIA Oil ─────────────────────────────────────────────────────
from src.data.ingestion.eia_ingestion import ingest_eia_with_fallback

print('[5/5] Ingesting EIA oil prices...')
eia_results = ingest_eia_with_fallback(start='2000-01-01')
count = get_row_count('raw', 'eia_oil')
print(f'✓ EIA oil: {eia_results} | raw.eia_oil total {count:,} rows')

[5/5] Ingesting EIA oil prices...
2026-05-05 00:29:14 | INFO     | src.data.ingestion.eia_ingestion | EIA series fetched
2026-05-05 00:29:14 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-05 00:29:14 | ERROR    | src.data.ingestion.eia_ingestion | Lỗi upsert EIA oil
Traceback (most recent call last):
  File "e:\study_again\gold_time_prediction\src\data\ingestion\eia_ingestion.py", line 121, in ingest_eia_with_fallback
    n = upsert_dataframe(df, table="eia_oil", schema=PG_SCHEMA_RAW, conflict_cols=["date", "series_id"])
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\study_again\gold_time_prediction\src\data\storage\postgres_client.py", line 144, in upsert_dataframe
    execute_values(cur, sql, rows, template=None, page_size=1000)
  File "c:\Users\Tuan\anaconda3\envs\std_again\Lib\site-packages\psycopg2\extras.py", line 1299, in execute_values
    cur.execute(b''.join(parts))
psycopg2.errors

In [10]:
# ── 6. Populate staging.daily_master ─────────────────────────────────────
from src.data.storage.postgres_client import execute_sql_file
from src.utils.config_loader import SQL_DIR

print('[6/7] Populating staging.daily_master...')
execute_sql_file(SQL_DIR / 'schema' / '00_populate_staging.sql')
count = get_row_count('staging', 'daily_master')
print(f'✓ staging.daily_master total: {count:,} rows')

[6/7] Populating staging.daily_master...
2026-05-05 00:34:30 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:34:30 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-05 00:34:30 | INFO     | src.data.storage.postgres_client | PG engine created
✓ staging.daily_master total: 0 rows


In [12]:
# ── 7. Cleaning ───────────────────────────────────────────────────────────
from src.data.preproccessing.cleaning import run_cleaning_pipeline

print('[7/7] Running cleaning pipeline...')
run_cleaning_pipeline(max_gap_days=3, z_threshold=5.0)
print('✓ Cleaning hoàn tất')

[7/7] Running cleaning pipeline...
2026-05-05 00:36:40 | INFO     | src.data.preproccessing.cleaning | === Bắt đầu Cleaning Pipeline ===
2026-05-05 00:36:40 | INFO     | src.data.preproccessing.cleaning | [1/3] Remove duplicates (raw tables)


2026-05-05 00:36:40 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: dedup raw.gold_prices
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: dedup raw.yfinance_daily
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: dedup raw.fred_daily
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: dedup raw.fred_monthly
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: dedup raw.eia_oil
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | [2/3] Fill missing (staging.daily_master)
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Forward-fill staging.daily_master
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | Cleaning SQL OK: ffill staging.daily_master
2026-05-05 00:36:41 | INFO     | src.data.preproccessing.cleaning | [3/3] Flag outliers (staging.daily_master)
2026-05-05 00:36:41 | INFO     

In [14]:
# ── 8. Feature Engineering ────────────────────────────────────────────────
from src.data.storage.postgres_client import run_feature_pipeline

print('[8/8] Running SQL feature engineering pipeline...')
run_feature_pipeline()

# Kiểm tra số dòng
for tbl in ['price_indicators', 'momentum_indicators', 'trend_indicators',
             'macro_features', 'ratio_features', 'sliding_windows',
             'target_labels', 'master_features']:
    count = get_row_count('features', tbl)
    print(f'  features.{tbl}: {count:,} rows')

[8/8] Running SQL feature engineering pipeline...
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | Bắt đầu pipeline [features]
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | [features] [1/8] Chạy: 01_price_features.sql
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | [features] [2/8] Chạy: 02_momentum_features.sql
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | [features] [3/8] Chạy: 03_trend_features.sql
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | [features] [4/8] Chạy: 04_macro_features.sql
2026-05-05 00:39:12 | INFO     | src.data.storage.postgres_client | SQL file executed
2026-05-05 00:39:12 | INFO     | src.data.storage.postg

In [15]:
# ── 9. Verification ───────────────────────────────────────────────────────
from sqlalchemy import text
from src.data.storage.postgres_client import get_engine

engine = get_engine()

# Sample từ master_features
with engine.connect() as conn:
    df_sample = pd.read_sql("""
        SELECT date, gold_close, sma_20, rsi_14, macd, adx_14,
               dxy_close, gold_silver_ratio, real_yield, gold_pct_chg_21d
        FROM features.master_features
        ORDER BY date DESC
        LIMIT 10
    """, conn)

print('=== Sample từ features.master_features (10 rows mới nhất) ===')
print(df_sample.to_string(index=False))

# Verify target labels KHÔNG có trong master_features
mf_cols = df_sample.columns.tolist()
leak_cols = ['next_1_day_price', 'next_3_day_price', 'next_7_day_price', 'next_30_day_price']
found_leaks = [c for c in leak_cols if c in mf_cols]
if found_leaks:
    print(f'⚠️  DATA LEAK DETECTED: {found_leaks}')
else:
    print('✓ Không có data leak trong master_features')

# Verify target labels gần đây nhất có NULL (đúng)
with engine.connect() as conn:
    df_targets = pd.read_sql("""
        SELECT date, next_1_day_price, next_7_day_price, next_30_day_price
        FROM features.target_labels
        ORDER BY date DESC
        LIMIT 5
    """, conn)
print('\n=== Target labels (5 rows mới nhất — phải có NULL) ===')
print(df_targets.to_string(index=False))

2026-05-05 00:39:18 | INFO     | src.data.storage.postgres_client | PG engine created
=== Sample từ features.master_features (10 rows mới nhất) ===
Empty DataFrame
Columns: [date, gold_close, sma_20, rsi_14, macd, adx_14, dxy_close, gold_silver_ratio, real_yield, gold_pct_chg_21d]
Index: []
✓ Không có data leak trong master_features

=== Target labels (5 rows mới nhất — phải có NULL) ===
Empty DataFrame
Columns: [date, next_1_day_price, next_7_day_price, next_30_day_price]
Index: []
